In [1]:
# import libraries 
import time
import re
import datetime
import csv
import logging
import sys
import os
import random

from selenium import webdriver
from bs4 import BeautifulSoup

# cần 2 cái dưới để điều hướng con trỏ chuột sang trang kế
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import logging # dùng để ghi tiến trình & lỗi khi crawl data

# Import 3 vũ khí mới để vượt qua bước Login của web
import os # để đọc thông tin trong file `.env`
import undetected_chromedriver as uc
from dotenv import load_dotenv

In [2]:
# --- Thiết lập logging ---
logging.basicConfig(
    filename="logging_careerviet.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding='utf-8'
)

# 1. Truy cập web

In [3]:
# https://careerviet.vn/viec-lam/Data-Analyst-k-vi.html
# https://careerviet.vn/viec-lam/Data-Analyst-k-trang-4-vi.html
# https://careerviet.vn/viec-lam/data-analyst-k-sortdv-trang-2-vi.html

KEYWORDS = ["Data Analyst"] # Rút gọn để test
BASE_URL_TEMPLATE = "https://careerviet.vn/viec-lam/{keyword}-k-sortdv-trang-{page}-vi.html"

MAX_PAGES = 3
OUTPUT_CSV = "data_careerviet.csv"

# ── 3. LOGGING (Giữ nguyên của bạn) ──────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")

In [4]:
# THAY ĐỔI 1: Dùng undetected_chromedriver thay vì webdriver.Chrome()
options = uc.ChromeOptions()


# Thêm cấu hình vô hiệu hóa chức năng hỏi lưu mật khẩu VÀ chặn thông báo (Notifications)
prefs = {
    "credentials_enable_service": False,
    "profile.password_manager_enabled": False,
    "profile.default_content_setting_values.notifications": 2  # Thêm dòng này! 2 có nghĩa là Block

}
options.add_experimental_option("prefs", prefs)



# THÊM version_main=140 VÀO ĐÂY vì nó mặc định chạy ver Chrome 147
driver = uc.Chrome(options=options, version_main=140) 


# Thay vì maximize_window(), dùng set_window_size và set_window_position
# Đặt kích thước cho trình duyệt (chiều rộng x chiều cao)
driver.set_window_size(900, 1080) # Bạn có thể thay đổi số theo độ phân giải màn hình
# (Tùy chọn) Đặt vị trí xuất hiện của cửa sổ (x, y) trên màn hình (0,0 là góc trên cùng bên trái)
driver.set_window_position(900, 0) # Có thể đẩy nó sang lề phải của màn hình

{'height': 882, 'width': 903, 'x': 900, 'y': 0}

# 4. Bắt đầu lấy data sản phẩm nè

1.	`job_url`: Link chi tiết của job đó.
2.	`job_title`: Tiêu đề công việc.
3.	`company_name`: Tên công ty.

4.	`salary`: Mức lương (thỏa thuận OR rank min - max).
5.	`location`: Địa điểm làm việc.
6.	`years_experience`: kinh nghiệm

7.	`job_deadline`: (Hạn nộp hồ sơ)
8.	`date_post`: Tính từ ngày dealine trừ đi 1 tháng

9.	`level_employee`: cấp bậc (Nhân viên)
10.	`job_type`: Hình thức làm việc ('nhân viên chính thức', t đoán chắc fulltime)

11.	`job_description`: Mô tả công việc.
12.	`requirements`: Yêu cầu công việc (Kinh nghiệm, kỹ năng).
13.	`benefits`: Phúc lợi (Lý do tại sao bạn sẽ yêu thích làm việc tại đây).
14. `other_info`: các thông tin về bằng cấp, lương, độ tuổi

In [92]:
# https://careerviet.vn/vi/tim-viec-lam/data-engineer.35C66D74.html
# job_link = 'https://careerviet.vn/vi/tim-viec-lam/data-analyst-internship.35C6F18D.html'
# job_link = 'https://careerviet.vn/vi/tim-viec-lam/data-analyst-trainee.35C6F17E.html'
job_link = 'https://careerviet.vn/vi/tim-viec-lam/data-analyst.35C6F18B.html'

# job_link = 'https://careerviet.vn/vi/tim-viec-lam/data-engineer.35C66D74.html'
job_link = job_link.split('?')[0]
job_link

'https://careerviet.vn/vi/tim-viec-lam/data-analyst.35C6F18B.html'

In [93]:

# Mở trang job chi tiết
driver.get(job_link)
time.sleep(random.uniform(3, 6)) # Đợi 3 giây để trang web tải hoàn toàn

# Cuộn xuống để load toàn bộ job
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

In [88]:
# Lấy mã nguồn HTML của trang web
new_html = driver.page_source
new_soup = BeautifulSoup(new_html, 'html.parser')

In [ ]:
new_soup

## 4.1 	Title Job

In [94]:
head_box = new_soup.find('div', class_='head-left')
head_box

<div class="head-left"><div class="caption"><div class="title" style="display:felx;align-items:center;gap:15px"><h2>Data Analyst</h2></div><a class="company" href="/vi/nha-tuyen-dung/careerviet.35A4E9A9.html" title="CareerViet">CareerViet</a><p class="salary"><em class="fa fa-usd"></em> <!-- -->Lương<!-- -->: <!-- -->Cạnh tranh</p></div></div>

In [95]:
title_tag = head_box.find('h2')
title_tag

<h2>Data Analyst</h2>

In [62]:
title_tag.get_text(separator=' ', strip=True)

'[Hà Nội] Business Intelligence Executive (Data Analyst Executive)'

In [96]:
# function 1: lấy tiêu đề job

def get_title(head_box):
    try:
        # 1. Tìm thẻ h2 trong head box, là tag duy nhất
        title_tag = head_box.find('h2')
            
        if title_tag:
            raw_title = title_tag.get_text(separator=' ', strip=True)
            return raw_title
                
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy title: {e}")
        return None

In [97]:
# test function: lấy tiêu đề job
title = get_title(head_box)
title

'Data Analyst'

## 4.2  Company name

In [15]:
# <a> class = company
company_tag = head_box.find('a', class_=re.compile(r'(?=.*company)', re.IGNORECASE))
company_tag

<a class="company" href="/vi/nha-tuyen-dung/wanek-furniture.35A5892F.html" title="Wanek Furniture">Wanek Furniture</a>

In [85]:
company_name = company_tag.get_text(strip=True)
company_name

'Wanek Furniture'

In [91]:
# function 6: lấy tên cty

def get_company(head_box):
    try:
        # <a> class = company
        company_tag = head_box.find('a', class_=re.compile(r'(?=.*company)', re.IGNORECASE))

        if company_tag:
            return company_tag.get_text(strip=True)

        return None
    except Exception:
        return None


company_name = get_company(head_box)
company_name

'Wanek Furniture'

##      Nguyên cái khối thông tin job  --> Dictionary

### 4.4. posted_date
### 4.5. deadline_date
### 4.6. category
### 4.7. work_type
### 4.8. salary
### 4.9. experience
### 4.10. level_employee

In [20]:
# class_='box-info' cho giá trị duy nhất 
box_info = new_soup.find('div', class_='box-info')
box_info

<div class="box-info"><div class="content"><p class="title">Thông Tin Tuyển Dụng</p><div class="table"><table><tbody><tr><td class="name"><p><em class="fa fa-id-badge"></em>Ngành nghề</p></td><td class="content"><span><a href="/viec-lam/cntt-phan-mem-c1-vi.html" title="CNTT - Phần mềm">CNTT - Phần mềm</a>, </span><span><a href="/viec-lam/san-xuat-van-hanh-san-xuat-c25-vi.html" title="Sản xuất / Vận hành sản xuất">Sản xuất / Vận hành sản xuất</a>, </span><span><a href="/viec-lam/do-go-c35-vi.html" title="Đồ gỗ">Đồ gỗ</a></span></td></tr><tr><td class="name"><p><em class="fa fa-usd"></em>Lương</p></td><td class="content"><p class="green"><strong>Cạnh tranh</strong></p></td></tr><tr><td class="name"><p><em class="mdi mdi-briefcase-edit"></em>Hình thức</p></td><td class="content"><p>Nhân viên chính thức</p></td></tr><tr><td class="name"><p><em class="mdi mdi-account"></em>Cấp bậc</p></td><td class="content"><p>Nhân viên</p></td></tr><tr><td class="name"><p><em class="fa fa-calendar-times-o

In [23]:
group_info = box_info.find_all('tr')
group_info

[<tr><td class="name"><p><em class="fa fa-id-badge"></em>Ngành nghề</p></td><td class="content"><span><a href="/viec-lam/cntt-phan-mem-c1-vi.html" title="CNTT - Phần mềm">CNTT - Phần mềm</a>, </span><span><a href="/viec-lam/san-xuat-van-hanh-san-xuat-c25-vi.html" title="Sản xuất / Vận hành sản xuất">Sản xuất / Vận hành sản xuất</a>, </span><span><a href="/viec-lam/do-go-c35-vi.html" title="Đồ gỗ">Đồ gỗ</a></span></td></tr>,
 <tr><td class="name"><p><em class="fa fa-usd"></em>Lương</p></td><td class="content"><p class="green"><strong>Cạnh tranh</strong></p></td></tr>,
 <tr><td class="name"><p><em class="mdi mdi-briefcase-edit"></em>Hình thức</p></td><td class="content"><p>Nhân viên chính thức</p></td></tr>,
 <tr><td class="name"><p><em class="mdi mdi-account"></em>Cấp bậc</p></td><td class="content"><p>Nhân viên</p></td></tr>,
 <tr><td class="name"><p><em class="fa fa-calendar-times-o"></em><span>Hết hạn nộp</span></p></td><td class="content"><p class="red">28/04/2026</p></td></tr>,
 <t

In [26]:
def get_general_info(box_info):
    """
    Quét các thẻ <tr> để lấy thông tin Key-Value và gom thành Dictionary.
    """
    info_dict = {}
    try:
        # 1. Tìm tất cả các dòng <tr>
        tr_tags = box_info.find_all('tr')
        
        for tr in tr_tags:
            # 2. Tìm tất cả các cột <td> bên trong dòng đó
            td_tags = tr.find_all('td')
            
            # Đảm bảo dòng này có đúng 2 cột (1 cột Name, 1 cột Content)
            if len(td_tags) == 2:
                
                # Lấy Key từ cột đầu tiên (Tự động bỏ qua cái icon <em>)
                key = td_tags[0].get_text(strip=True)
                
                # Lấy Value từ cột thứ hai
                # Dùng separator=" " để các ngành nghề không bị dính chữ vào nhau
                raw_value = td_tags[1].get_text(separator=" ", strip=True)
                
                # Làm sạch nhẹ: Xóa khoảng trắng thừa trước dấu phẩy ở phần Ngành nghề
                clean_value = raw_value.replace(" ,", ",")
                
                # Lưu vào Dictionary
                info_dict[key] = clean_value
                
        return info_dict

    except Exception as e:
        logging.error(f"Lỗi khi lấy thông tin chung VIP: {e}")
        return info_dict

# ================= ÁP DỤNG =================
# Giả sử 'box_info' là thẻ div chứa cái bảng đó
job_info = get_general_info(box_info)
print(*(f"{k}: {v}" for k, v in job_info.items()), sep="\n")

posted_date     = job_info.get('Ngày cập nhật')
deadline_date   = job_info.get('Hết hạn nộp')
category        = job_info.get('Ngành nghề')
work_type       = job_info.get('Hình thức')
salary          = job_info.get('Lương')
experience      = job_info.get('Kinh nghiệm')
level_employee  = job_info.get('Cấp bậc')
location        = job_info.get('Địa điểm')

Ngành nghề: CNTT - Phần mềm, Sản xuất / Vận hành sản xuất, Đồ gỗ
Lương: Cạnh tranh
Hình thức: Nhân viên chính thức
Cấp bậc: Nhân viên
Hết hạn nộp: 28/04/2026
Ngày cập nhật: 30/03/2026
Địa điểm: Bình Dương


In [28]:
posted_date

'30/03/2026'

In [29]:
deadline_date

'28/04/2026'

In [30]:
category

'CNTT - Phần mềm, Sản xuất / Vận hành sản xuất, Đồ gỗ'

In [31]:
work_type

'Nhân viên chính thức'

In [32]:
salary

'Cạnh tranh'

In [33]:
experience

In [34]:
level_employee

'Nhân viên'

In [35]:
location

'Bình Dương'

### 4.11        location

In [38]:
# class = detail-row info-place-detail
box_location = new_soup.find('div', class_='detail-row info-place-detail')
box_location

<div class="detail-row info-place-detail"><h3 class="detail-title">Địa điểm làm việc</h3><div class="place-name"><div class="place">Bình Dương</div><span><i class="fa fa-map-marker fa-fw"></i>So 11, Duong so 6, KCN VSIP II, Thu Dau Mot, Binh Duong</span></div></div>

In [40]:
location_tag = box_location.find('div', class_='place')
location_tag

<div class="place">Bình Dương</div>

In [49]:
location = location_tag.get_text(strip=True)
location

'Bình Dương'

In [50]:
# Hàm lấy địa điểm làm việc

def get_location(new_soup):
    """
    Tìm và lấy trực tiếp thông tin "Địa điểm" trong khối box bên dưới.
    """
    try:
        # 1. Test xem có cái box Dictionary có thông tin Location ko
        location_dict = job_info.get('Địa điểm')

        if not location_dict:
            # class = detail-row info-place-detail
            box_location = new_soup.find('div', class_='detail-row info-place-detail')

            
            # 2. Lấy tag location
            location_tag = box_location.find('div', class_='place')
            

            if location_tag:
                return  location_tag.get_text(strip=True) 
        
        else:
            return location_dict
        

    except Exception as e:
        logging.error(f"Lỗi khi lấy Địa điểm: {e}")
        return None

# ================= ÁP DỤNG =================
location = get_location(new_soup)
location

'Bình Dương'

## Nguyên Cái Khối Dưới

### 4.11       education_level

In [98]:
# 1. Tìm cái box "Thông tin khác" bên dưới cùng, chỉ ra 1 kết quả duy nhất
other_box_title = new_soup.find('h3', class_="detail-title", string='Thông tin khác')
other_box_title

<h3 class="detail-title">Thông tin khác</h3>

In [101]:
other_box = other_box_title.parent
other_box

<div class="detail-row"><h3 class="detail-title">Thông tin khác</h3><div class="content"><ul><li>Bằng cấp<!-- -->:<!-- --> <!-- -->Đại học</li><li>Giới tính<!-- -->:<!-- --> <!-- -->Nam</li> <li>Hình thức<!-- -->:<!-- --> <!-- -->Nhân viên chính thức</li> <li>Độ tuổi<!-- -->:<!-- --> <!-- -->Không giới hạn tuổi</li><li>Lương<!-- -->: <!-- -->Cạnh tranh</li></ul></div></div>

In [103]:
other_box.find('li', string=re.compile(r'Bằng cấp', re.IGNORECASE))

In [106]:
# 2. Lấy tag bằng cấp
edu_tag = other_box.find(
    lambda tag: tag.name == 'li' and 'Bằng cấp' in tag.get_text(strip=True)
)
edu_tag

<li>Bằng cấp<!-- -->:<!-- --> <!-- -->Đại học</li>

In [107]:
# 3. Lấy text
raw_text = edu_tag.get_text(strip=True) # VD: "Bằng cấp: Đại học"
raw_text

'Bằng cấp:Đại học'

In [108]:
# 4. Xử lý text
if ':' in raw_text:
    # Lấy phần tử thứ 2 (index 1) sau dấu ':' và xóa khoảng trắng
    education_level = raw_text.split(':', 1)[1].strip() 


education_level

'Đại học'

In [109]:


def get_education_level(new_soup):
    """
    Tìm và lấy trực tiếp thông tin "Bằng cấp" trong khối 'Thông tin khác'.
    """
    try:
        # 1. Tìm cái box "Thông tin khác" bên dưới cùng, chỉ ra 1 kết quả duy nhất
        other_box_title = new_soup.find('h3', class_="detail-title", string='Thông tin khác')

        if other_box_title:
            other_box = other_box_title.parent

            
            # 2. Lấy tag bằng cấp
            edu_tag = other_box.find(
                lambda tag: tag.name == 'li' and 'Bằng cấp' in tag.get_text(strip=True)
            )
            

            if edu_tag:
                raw_text = edu_tag.get_text(strip=True) # VD: "Bằng cấp: Đại học"
                
                # 3. Cắt chuỗi lấy phần Value
                if ':' in raw_text:
                    # Lấy phần tử thứ 2 (index 1) sau dấu ':' và xóa khoảng trắng
                    return raw_text.split(':', 1)[1].strip() 
                    
        return None

    except Exception as e:
        logging.error(f"Lỗi khi lấy Bằng cấp: {e}")
        return None

# ================= ÁP DỤNG =================
education_level = get_education_level(new_soup)
education_level

'Đại học'

### 4.12      MÔ TẢ CÔNG VIỆC
### 4.13      YÊU CẦU CÔNG VIỆC
### 4.14      PHÚC LỢI

In [80]:
# 1. Tìm thẻ h3 chứa từ khóa (Mỏ neo) - Không phân biệt hoa thường
# t phát hiện ra 1 tác dụng của Regex là nó ko phân biệt viết HOA - thường
h2_tag = new_soup.find(
    ['h2', 'h3'], 
    class_='detail-title', 
    string=re.compile('Mô tả Công việc', re.IGNORECASE)
    )
h2_tag

<h2 class="detail-title">Mô tả Công việc</h2>

In [72]:
# 2. Lấy tất cả các thẻ nằm SAU thẻ h3 này (p, ul, li...)
h2_tag.find_next_siblings()

[<div class="content_fck custom-job-des_customJobDes__C5nnx"><ul><li>Collaborate with AI/ML Engineers to design and maintain data pipelines (ETL/ELT) from ERP, IoT, and production systems.</li><li>Clean, transform, and structure data for machine learning and analytics.</li><li>Manage data storage, processing, and retrieval in Data Warehouse or Data Lake environments.</li><li>Conduct exploratory data analysis and statistical modeling to support AI initiatives.</li><li>Ensure data quality, integrity, and compliance with company standards.</li><li>Translate business requirements into data-driven solutions in collaboration with IT and business stakeholders.</li><li>Monitor and optimize data workflows for performance and scalability.</li><li>Document data processes and model workflows for maintainability.</li></ul></div>]

In [74]:
content_parts = []

for sibling in h2_tag.find_next_siblings():
    # Lấy text, dùng dấu cách để các chữ không bị dính vào nhau
    text = sibling.get_text(separator='. ', strip=True)
    if text:
        content_parts.append(text)

content_parts        

['Collaborate with AI/ML Engineers to design and maintain data pipelines (ETL/ELT) from ERP, IoT, and production systems.. Clean, transform, and structure data for machine learning and analytics.. Manage data storage, processing, and retrieval in Data Warehouse or Data Lake environments.. Conduct exploratory data analysis and statistical modeling to support AI initiatives.. Ensure data quality, integrity, and compliance with company standards.. Translate business requirements into data-driven solutions in collaboration with IT and business stakeholders.. Monitor and optimize data workflows for performance and scalability.. Document data processes and model workflows for maintainability.']

In [75]:
# 3. Nối tất cả lại thành 1 đoạn văn duy nhất
# Đã check, và nó cào toàn bộ nội dung phần Mô tả thật
raw_text = ' '.join(content_parts)
raw_text

'Collaborate with AI/ML Engineers to design and maintain data pipelines (ETL/ELT) from ERP, IoT, and production systems.. Clean, transform, and structure data for machine learning and analytics.. Manage data storage, processing, and retrieval in Data Warehouse or Data Lake environments.. Conduct exploratory data analysis and statistical modeling to support AI initiatives.. Ensure data quality, integrity, and compliance with company standards.. Translate business requirements into data-driven solutions in collaboration with IT and business stakeholders.. Monitor and optimize data workflows for performance and scalability.. Document data processes and model workflows for maintainability.'

In [76]:

    
# Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)

# Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
final_text = ' '.join(clean_text.split())

# Chống lỗi công thức khi xuất ra file CSV/Excel
if final_text.startswith(('-', '=', '+', '@')):
    final_text = "'" + final_text


final_text

'Collaborate with AI/ML Engineers to design and maintain data pipelines (ETL/ELT) from ERP, IoT, and production systems.. Clean, transform, and structure data for machine learning and analytics.. Manage data storage, processing, and retrieval in Data Warehouse or Data Lake environments.. Conduct exploratory data analysis and statistical modeling to support AI initiatives.. Ensure data quality, integrity, and compliance with company standards.. Translate business requirements into datadriven solutions in collaboration with IT and business stakeholders.. Monitor and optimize data workflows for performance and scalability.. Document data processes and model workflows for maintainability.'

In [78]:
# Cào cùng lúc 3 mục, vì ở trong cùng 1 box

def extract_paragraph_content(new_soup, heading_keyword):
    """
    Hàm đa năng: Tìm thẻ h2 chứa keyword, sau đó lấy toàn bộ text 
    của các thẻ anh em (siblings) nằm dưới nó trong cùng 1 div.
    """
    try:
        # 1. thẻ title của box:
        pattern_string = re.compile(rf'{heading_keyword}', re.IGNORECASE)
        h2_tag = new_soup.find(['h2', 'h3'], class_='detail-title', string=pattern_string)


        # 2. Tìm thẻ cha là BOX chứa cái thông tin đó
        if h2_tag:

            # 2. Lấy nguyên text
            content_parts = []

            for sibling in h2_tag.find_next_siblings():
                # Lấy text, dùng '. ' làm dải phân cách cho các thẻ con (như <li>, <br>)
                text = sibling.get_text(separator='. ', strip=True)
                if text:
                    content_parts.append(text)



            # 3. Nối tất cả lại thành 1 đoạn văn duy nhất
            # Đã check, và nó cào toàn bộ nội dung phần Mô tả thật
            raw_text = ' '.join(content_parts)


            # 4. Làm sạch text
            # Xóa các ký tự bullet point thường gặp (Thêm dấu '-' vì nhiều HR dùng gạch đầu dòng)
            clean_text = re.sub(r'[•\*●\-\+\−]', '', raw_text)
            
            # Tuyệt chiêu gom khoảng trắng: Xóa sạch \n, \t, \xa0 và các dấu cách thừa
            final_text = ' '.join(clean_text.split())
            
            # Chống lỗi công thức khi xuất ra file CSV/Excel
            if final_text.startswith(('-', '=', '+', '@')):
                final_text = "'" + final_text
        
            return final_text
            
        return None
    except Exception as e:
        logging.error(f"Lỗi khi lấy {heading_keyword}: {e}")
        return None



# ==========================================
# ÁP DỤNG VÀO 3 HÀM CỦA BẠN
# ==========================================

def get_job_description(new_soup):
    return extract_paragraph_content(new_soup, "Mô tả công việc")

def get_job_requirements(new_soup):
    return extract_paragraph_content(new_soup, "Yêu Cầu Công Việc")

def get_benefits(new_soup):
    return extract_paragraph_content(new_soup, "Phúc lợi")

In [82]:
description = get_job_description(new_soup)
description

'Collaborate with AI/ML Engineers to design and maintain data pipelines (ETL/ELT) from ERP, IoT, and production systems.. Clean, transform, and structure data for machine learning and analytics.. Manage data storage, processing, and retrieval in Data Warehouse or Data Lake environments.. Conduct exploratory data analysis and statistical modeling to support AI initiatives.. Ensure data quality, integrity, and compliance with company standards.. Translate business requirements into datadriven solutions in collaboration with IT and business stakeholders.. Monitor and optimize data workflows for performance and scalability.. Document data processes and model workflows for maintainability.'

In [83]:
requirements = get_job_requirements(new_soup)
requirements

'Bachelor’s degree in Computer Science, Information Technology, Data Science, or a related field.. Minimum 23 years. of handson experience in. Data engineering, Data integration. , or a similar datafocused role.. Strong proficiency in. SQL. and. Python. , with experience using data manipulation libraries such as. Pandas. and. PySpark. .. Proven experience in designing and implementing. ETL/ELT pipelines. and automating data workflows.. Familiarity with cloud platforms such as. Azure. ,. AWS. , or. Google Cloud Platform (GCP). , including cloudbased data storage and processing tools.. Experience working with. APIs. ,. relational and NoSQL databases. , and. largescale data processing systems. .. Solid understanding of. data modeling. ,. schema design. , and. data governance. best practices.. Strong analytical and problemsolving skills, with the ability to work both independently and in crossfunctional teams.. Good English communication.'

In [84]:
benefits = get_benefits(new_soup)
benefits

'Laptop. Chế độ bảo hiểm. Du Lịch. Phụ cấp. Xe đưa đón. Đồng phục. Chế độ thưởng. Chăm sóc sức khỏe. Đào tạo. Tăng lương. Công tác phí. Nghỉ phép năm. CLB thể thao'